<- [19 - Solver Choice and Tactics](19_Solver_Choice_And_Tactics.ipynb) | [README Z3](README.md)

***

# 20 — Étendre le DSL : réécriture de prédicats et réécriture globale

> Notebook de la série **Z3 / SMT** (track C#, fork [Z3.Linq](../Z3.Linq/)).
> Capacité **jamais exercée dans les carnets de la série** — vérifié par grep des deux
> attributs sur les carnets 01-18 (zéro occurrence) ; couverte uniquement par les tests
> unitaires du fork (`Z3.Linq.Tests/RewriterTests.cs`), jamais montrée en notebook :
> les **points d'extension de réécriture** — `[TheoremPredicateRewriter]` et
> `[TheoremGlobalRewriter]` — qui permettent d'adapter le DSL à un vocabulaire métier
> **sans modifier la bibliothèque**.

## Le problème

Un DSL n'est jamais complet. Le traducteur de Z3.Linq sait traduire l'arithmétique, les
comparaisons, les tableaux, les `record`… mais pas *vos* méthodes. Que faire quand une contrainte
s'écrit naturellement `t => TousDifferents(t.A, t.B, t.C)` alors que `TousDifferents` n'a aucun
sens pour Z3 ? Trois issues :

1. inliner la contrainte à la main partout — répétition, perte du vocabulaire métier ;
2. forker la bibliothèque et ajouter un cas au traducteur — lourd, irreproductible ;
3. **brancher un rewriter** : laisser la bibliothèque appeler *votre* code au moment où elle
   rencontre la construction inconnue, et lui rendre une expression qu'elle sait traduire.

C'est le troisième chemin que ce carnet démontre, aux deux granularités offertes par le fork :
**au niveau d'un appel de méthode** (prédicat) et **au niveau de l'ensemble des contraintes**
(global).

## Ce que le carnet mesure, et pas seulement affirme

Chaque mécanisme est jugé par ce qu'il **change au verdict du solveur** :

- un prédicat métier refusé par le traducteur devient résoluble **grâce** au rewriter ;
- un rewriter **identité** est refusé par la bibliothèque (garde anti-boucle) ;
- une cible de réécriture **non traduisible** est refusée aussi — la contrainte de conception
  du mécanisme, mesurée et non supposée ;
- un rewriter **global** qui supprime une contrainte renverse un `UNSAT` en `SAT`, et
  `Explain()` rapporte la contrainte **injectée** par le rewriter — la cohérence
  diagnostic / solveur est montrée, pas affirmée.
- un rewriter **global** à l'échelle : sur un Sudoku de **138 contraintes**, une seule règle
  ajoutée rend le théorème insoluble — le cœur UNSAT nomme la règle (#138) en face de l'indice
  contredit (#108) ;
- la **garde de formation** : un attribut qui désigne un type n'implémentant pas l'interface
  attendue est refusé, côté prédicat comme côté global, avec le message du binding.

## Ce que ce carnet n'est pas

Il ne ré-implémente pas le traducteur (organ-first : l'organe est dans le fork, le carnet
l'invoque), ne corrige pas la bibliothèque, et ne couvre ni `SolverKind` / `.Logic` / `.Tactics`
(carnet 19), ni les quantificateurs bornés `ForAll` / `Exists` (traités ailleurs dans la série).


## La surface publique : deux attributs, deux points d'appel

Le fork expose ces deux mécanismes par des **attributs** — c'est toute la surface publique :

| Attribut | Porte sur | Effet |
|---|---|---|
| `[TheoremGlobalRewriter(typeof(R))]` | le **type d'environnement** | `R` reçoit toutes les contraintes du théorème et renvoie la liste à asserter |
| `[TheoremPredicateRewriter(typeof(R))]` | une **méthode de prédicat** | `R` reçoit l'appel de méthode et renvoie l'appel à traduire à sa place |

Mesure de l'absence, rejouable, sur `origin/main` avant ce carnet :

```
git grep -o -E "ITheoremGlobalRewriter|ITheoremPredicateRewriter|TheoremGlobalRewriter|TheoremPredicateRewriter" \
    origin/main -- 'MyIA.AI.Notebooks/**/*.ipynb' | wc -l
0
```

Le fork implémente pourtant cette surface, fichiers et points d'appel citables : les interfaces et leurs attributs (`solutions/Z3.Linq/ITheoremGlobalRewriter.cs`, `ITheoremPredicateRewriter.cs`, `TheoremGlobalRewriterAttribute.cs`, `TheoremPredicateRewriterAttribute.cs`) et les deux points d'appel — `Theorem.cs:396-418` (`GetConstraintsToAssert`, partagé par le chemin de résolution et par `Explain`) et `ExpressionVisitor.cs:446-471` (`VisitCall`, avec la garde anti-boucle). C'est une capacité **publique, réelle et jamais exposée** en carnet avant celui-ci.


In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Linq;
using System.Linq.Expressions;
using System.Collections.Generic;
using System.Text;

Console.OutputEncoding = Encoding.UTF8;
Console.WriteLine("Imports OK : Z3.Linq (.deploy du fork) + Microsoft.Z3 + arbres d'expression");
Console.WriteLine($"Point d'extension 1 : {typeof(ITheoremPredicateRewriter).FullName}");
Console.WriteLine($"Point d'extension 2 : {typeof(ITheoremGlobalRewriter).FullName}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq (.deploy du fork) + Microsoft.Z3 + arbres d'expression


Point d'extension 1 : Z3.Linq.ITheoremPredicateRewriter


Point d'extension 2 : Z3.Linq.ITheoremGlobalRewriter


## 1. Le point de blocage, mesuré sans rewriter

Un triplet d'entiers `(A, B, C)` devant prendre trois valeurs **deux à deux distinctes**, positives,
de somme 6 : les solutions sont les permutations de `(1, 2, 3)`. En C# métier, la contrainte
s'écrit `TousDifferents(A, B, C)`.

Confrontons cette écriture au traducteur, sans aucune aide.


In [2]:
// Un predicat metier, volontairement SANS attribut de reecriture.
public static class PredicatsSansAttribut
{
    public static bool TousDifferents(int a, int b, int c) => a != b && b != c && a != c;
}

public class Triplet
{
    public int A { get; set; }
    public int B { get; set; }
    public int C { get; set; }
}

using (var ctx = new Z3Context())
{
    try
    {
        var theorem = from t in ctx.NewTheorem<Triplet>()
                      where PredicatsSansAttribut.TousDifferents(t.A, t.B, t.C)
                      where t.A > 0 && t.B > 0 && t.C > 0
                      where t.A + t.B + t.C == 6
                      select t;
        var sol = theorem.Solve();
        Console.WriteLine($"INATTENDU : solution trouvee -> {sol}");
    }
    catch (Exception ex)
    {
        Console.WriteLine($"REFUS attendu : {ex.GetType().Name}");
        Console.WriteLine($"  {ex.Message}");
    }
}


REFUS attendu : NotSupportedException


  Unknown method call:Boolean TousDifferents(Int32, Int32, Int32)



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture du résultat

Le traducteur ne devine pas qu'un appel de méthode signifie autre chose. Sa table de traduction
couvre des **nœuds d'expression** (arithmétique, comparaisons, conversions) et une liste close de
méthodes « magiques » ; tout le reste lève `NotSupportedException`.

C'est exactement le trou que comble le rewriter — et non un contournement par « inliner à la main
partout », qui ferait perdre le vocabulaire métier dans chaque contrainte.


## 2. Le contrat du rewriter de prédicat

Lu dans le fork (`ITheoremPredicateRewriter.cs`, `ExpressionVisitor.VisitCall`) :

| Règle | Portée |
|---|---|
| Attribut `[TheoremPredicateRewriter(typeof(...))]` posé sur **une méthode** | c'est l'**appel** de cette méthode qui déclenche la réécriture |
| Constructeur **public sans paramètre** du rewriter | la bibliothèque l'instancie par `Activator.CreateInstance` |
| `Rewrite(MethodCallExpression call)` rend un **`MethodCallExpression`** | pas n'importe quelle `Expression` : un *appel* |
| Le résultat doit **différer** de l'entrée | sinon `InvalidOperationException` immédiate — garde anti-boucle |

### La contrainte de conception, vérifiée à la source

`VisitCall` ne fait **pas** d'inlining de corps de méthode : après la réécriture, l'expression
rendue est re-visitée telle quelle, et si elle n'appartient pas à la table de traduction, elle est
refusée à son tour. **Un rewriter ne peut donc pas se contenter de « déplier » vers du C# arbitraire :
sa cible doit être une forme que le traducteur sait déjà traduire.** Dans le fork, ces formes sont
les opérateurs « magiques » de `Z3Methods` — `Distinct`, `Sum`, `ExactlyOne`, `AtMostOne`,
`AtLeastOne`, `Weighted*`, `ForAll`, `Exists` — et les accesseurs indexés.

C'est ce qui rend le mécanisme utile : on habille d'un vocabulaire métier des opérateurs que le
traducteur traite déjà, mais que personne n'a envie d'écrire à la main dans une contrainte.
La section 4 en fait la démonstration négative.


In [3]:
// Rewriters : chacun rend un appel que le traducteur sait deja traiter.
public class DeplieurMetier : ITheoremPredicateRewriter
{
    private static readonly System.Reflection.MethodInfo DistinctInt =
        typeof(Z3Methods).GetMethod(nameof(Z3Methods.Distinct))!.MakeGenericMethod(typeof(int));

    private static readonly System.Reflection.MethodInfo ExactlyOneBool =
        typeof(Z3Methods).GetMethod(nameof(Z3Methods.ExactlyOne))!;

    public MethodCallExpression Rewrite(MethodCallExpression call)
    {
        var args = call.Arguments;
        switch (call.Method.Name)
        {
            case nameof(PredicatsMetier.TousDifferents):
                // TousDifferents(a, b, c)  =>  Z3Methods.Distinct(a, b, c)   (params int[])
                return Expression.Call(DistinctInt, Expression.NewArrayInit(typeof(int), args));
            case nameof(PredicatsMetier.UneSeuleVraie):
                // UneSeuleVraie(p, q)  =>  Z3Methods.ExactlyOne(p, q)        (params bool[])
                return Expression.Call(ExactlyOneBool, Expression.NewArrayInit(typeof(bool), args));
            default:
                throw new InvalidOperationException($"Predicat metier inconnu : {call.Method.Name}");
        }
    }
}

// Temoin negatif 1 : renvoie l'appel recu -> rejet anti-boucle de la bibliotheque.
public class RewriterIdentite : ITheoremPredicateRewriter
{
    public MethodCallExpression Rewrite(MethodCallExpression call) => call;
}

// Temoin negatif 2 : deplie vers une methode ordinaire, que le traducteur ne connait pas.
public class DeplieurNaif : ITheoremPredicateRewriter
{
    public MethodCallExpression Rewrite(MethodCallExpression call)
    {
        var cible = typeof(FormesNonTraduisibles).GetMethod(nameof(FormesNonTraduisibles.Congru));
        return Expression.Call(cible!, call.Arguments[0], Expression.Constant(2), Expression.Constant(0));
    }
}

public static class FormesNonTraduisibles
{
    public static bool Congru(int valeur, int modulo, int reste) => valeur % modulo == reste;
}

// Le vocabulaire metier, annote : c'est l'appel de CES methodes qui declenche la reecriture.
public static class PredicatsMetier
{
    [TheoremPredicateRewriter(typeof(DeplieurMetier))]
    public static bool TousDifferents(int a, int b, int c) => a != b && b != c && a != c;

    [TheoremPredicateRewriter(typeof(DeplieurMetier))]
    public static bool UneSeuleVraie(bool p, bool q) => p ^ q;

    [TheoremPredicateRewriter(typeof(RewriterIdentite))]
    public static bool Boucle(int x) => true;

    [TheoremPredicateRewriter(typeof(DeplieurNaif))]
    public static bool EstPair(int x) => x % 2 == 0;
}

Console.WriteLine("Definitions pretes : DeplieurMetier (2 predicats), RewriterIdentite, DeplieurNaif, PredicatsMetier");


Definitions pretes : DeplieurMetier (2 predicats), RewriterIdentite, DeplieurNaif, PredicatsMetier



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

## 3. Ce que la réécriture change au verdict

Même théorème que la section 1 — mêmes contraintes écrites, un seul changement : la méthode porte
l'attribut. Si le rewriter est bien celui décrit, la résolution aboutit et le témoin vérifie la
propriété **en C#**, hors de Z3.


In [4]:
// PREUVE 1 : le predicat metier est traduit, le theoreme est resolu.
using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<Triplet>()
                  where PredicatsMetier.TousDifferents(t.A, t.B, t.C)
                  where t.A > 0 && t.B > 0 && t.C > 0
                  where t.A + t.B + t.C == 6
                  select t;
    var sol = theorem.Solve();
    Console.WriteLine($"« TousDifferents(A,B,C), A,B,C>0, A+B+C=6 » -> {(sol == null ? "UNSAT" : $"(A,B,C) = ({sol.A},{sol.B},{sol.C})")}");
    if (sol != null)
    {
        bool somme = sol.A + sol.B + sol.C == 6;
        bool positifs = sol.A > 0 && sol.B > 0 && sol.C > 0;
        bool distincts = sol.A != sol.B && sol.B != sol.C && sol.A != sol.C;
        Console.WriteLine($"Temoin verifie hors Z3 : somme={somme}, positifs={positifs}, distincts={distincts}");
        Console.WriteLine($"Verdict : {(somme && positifs && distincts ? "solution valide" : "solution INVALIDE")}");
    }
}


« TousDifferents(A,B,C), A,B,C>0, A+B+C=6 » -> (A,B,C) = (3,1,2)


Temoin verifie hors Z3 : somme=True, positifs=True, distincts=True


Verdict : solution valide



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

In [5]:
// PREUVE 2 : second predicat, sur un environnement booleen, avec une reecriture differente.
public class PaireBooleenne
{
    public bool P { get; set; }
    public bool Q { get; set; }
}

using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<PaireBooleenne>()
                  where PredicatsMetier.UneSeuleVraie(t.P, t.Q)
                  where !t.P
                  select t;
    var sol = theorem.Solve();
    Console.WriteLine($"« UneSeuleVraie(P,Q), !P » -> {(sol == null ? "UNSAT" : $"(P,Q) = ({sol.P},{sol.Q})")}");
    Console.WriteLine("Attendu : P = False, Q = True (exactement un des deux)");
    Console.WriteLine("Temoin verifie hors Z3 : " + (sol != null && sol.P == false && sol.Q == true));
}


« UneSeuleVraie(P,Q), !P » -> (P,Q) = (False,True)


Attendu : P = False, Q = True (exactement un des deux)


Temoin verifie hors Z3 : True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



## 4. Les deux gardes, mesurées

Un mécanisme d'extension sans garde-fou est une invitation à la boucle infinie. Le fork en pose
plusieurs ; cette section en déclenche exprès deux : le rewriter **identité** (qui ne réécrit
rien) et le rewriter **naïf** (qui déplie vers une méthode ordinaire, non traduisible). La
troisième — un attribut qui désigne un type n'implémentant pas l'interface attendue — attend la
section 8 : son témoin côté rewriter global est un type d'environnement Sudoku, défini en
section 7.


In [6]:
// TEMOIN NEGATIF 1 : rewriter identite -> rejet anti-boucle.
using (var ctx = new Z3Context())
{
    try
    {
        var theorem = from t in ctx.NewTheorem<Triplet>()
                      where PredicatsMetier.Boucle(t.A)
                      select t;
        Console.WriteLine($"INATTENDU : resolution aboutie -> {theorem.Solve()}");
    }
    catch (InvalidOperationException ex)
    {
        Console.WriteLine("REJET attendu (InvalidOperationException) :");
        Console.WriteLine("  " + ex.Message);
        bool gardeNommee = ex.Message.Contains("did not perform any rewrite");
        Console.WriteLine("Mesure : le message nomme la non-reecriture = " + gardeNommee);
        if (!gardeNommee)
            throw new InvalidOperationException(
                "Temoin 1 : la garde anti-boucle n'est pas nommee dans le message (attendu : did not perform any rewrite).");
    }
}


REJET attendu (InvalidOperationException) :


  The expression tree rewriter of type RewriterIdentite did not perform any rewrite. Aborting compilation to avoid infinite looping.


Mesure : le message nomme la non-reecriture = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



In [7]:
// TEMOIN NEGATIF 2 : deplier vers une methode ordinaire ne suffit pas.
// Le traducteur ne fait pas d'inlining de corps : la cible doit etre une forme traduisible.
using (var ctx = new Z3Context())
{
    try
    {
        var theorem = from t in ctx.NewTheorem<Triplet>()
                      where PredicatsMetier.EstPair(t.A)
                      select t;
        Console.WriteLine($"INATTENDU : resolution aboutie -> {theorem.Solve()}");
    }
    catch (NotSupportedException ex)
    {
        Console.WriteLine("REFUS attendu (NotSupportedException) :");
        Console.WriteLine("  " + ex.Message);
        bool refusSurCible = ex.Message.StartsWith("Unknown method call");
        Console.WriteLine("Mesure : le refus porte sur l'appel CIBLE, pas sur le predicat d'origine = " + refusSurCible);
        if (!refusSurCible)
            throw new InvalidOperationException(
                "Temoin 2 : le refus devait porter sur l'appel CIBLE (Unknown method call), message observe : " + ex.Message);
    }
}


REFUS attendu (NotSupportedException) :


  Unknown method call:Boolean Congru(Int32, Int32, Int32)


Mesure : le refus porte sur l'appel CIBLE, pas sur le predicat d'origine = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



### Lecture du résultat

Les deux refus sont complémentaires, et c'est le second qui porte la leçon de conception :

- **identité** → la bibliothèque refuse une réécriture qui n'avance pas (terminaison garantie) ;
- **cible non traduisible** → la réécriture a bien eu lieu, mais l'expression rendue est re-visitée
  telle quelle et rejetée, parce que le traducteur ne connaît pas `Congru`.

Autrement dit : le rewriter **ne fait pas de macro-expansion de C#**. Il traduit un vocabulaire
métier vers la **grammaire du traducteur**, pas vers du code arbitraire. C'est une contrainte à
connaître avant d'en écrire un — et elle explique pourquoi la cible utile est presque toujours un
opérateur de `Z3Methods`.

La correspondance est donc déclarative : `TousDifferents(a,b,c)` **est** `Distinct(a,b,c)`,
`UneSeuleVraie(p,q)` **est** `ExactlyOne(p,q)`. Le carnet garde le mot métier, le solveur reçoit
l'opérateur qu'il sait propager.


## 5. Le rewriter global : réécrire l'ensemble des contraintes

Le second mécanisme change d'échelle. Lu dans le fork (`Theorem.GetConstraintsToAssert`,
`Theorem.Explain`) :

- on annote le **type d'environnement** avec `[TheoremGlobalRewriter(typeof(...))]` ;
- à chaque résolution, la bibliothèque instancie le rewriter et lui passe **toutes** les contraintes
  accumulées, dans l'ordre de leur déclaration ;
- le rewriter rend les contraintes **à asserter à la place** : il peut en ajouter, en supprimer,
  en remplacer.

Conséquence architecturale, écrite dans le fork : `GetConstraintsToAssert` est **partagé par le
chemin de résolution et par `Explain`**. Le diagnostic porte donc sur les contraintes
**réécrites** — pas sur celles que vous avez écrites. C'est la cohérence voulue : rapporter un
conflit sur des contraintes que le solveur n'a jamais vues serait un mensonge de diagnostic.

Premier effet, le plus net : une contrainte **supprimée** rend satisfiable un système qui ne
l'était pas.


In [8]:
// Environnement SANS rewriter : systeme contradictoire (X > 5 et X < 3).
public class SansRewriter
{
    public int X { get; set; }
}

// Rewriter global : ecarte les contraintes dont la forme textuelle contient "t.X < 3".
// Filtre volontairement simple et documente : c'est une demonstration de mecanisme,
// pas une heuristique de production.
public class RelaxBorneHaute : ITheoremGlobalRewriter
{
    public IEnumerable<LambdaExpression> Rewrite(IEnumerable<LambdaExpression> constraints)
    {
        var recues = constraints.ToList();
        Console.WriteLine($"  [rewriter global] {recues.Count} contrainte(s) recue(s)");
        var conservees = recues.Where(c => !c.Body.ToString().Contains("t.X < 3")).ToList();
        Console.WriteLine($"  [rewriter global] {recues.Count - conservees.Count} supprimee(s), {conservees.Count} conservee(s)");
        return conservees;
    }
}

[TheoremGlobalRewriter(typeof(RelaxBorneHaute))]
public class AvecRelaxation
{
    public int X { get; set; }
}

using (var ctx = new Z3Context())
{
    var dur = from t in ctx.NewTheorem<SansRewriter>()
              where t.X > 5
              where t.X < 3
              select t;
    var solDur = dur.Solve();
    Console.WriteLine($"SANS rewriter : « X>5, X<3 » -> {(solDur == null ? "UNSAT (null)" : $"X = {solDur.X}")}");

    Console.WriteLine("AVEC rewriter global sur les memes contraintes ecrites :");
    var relax = from t in ctx.NewTheorem<AvecRelaxation>()
                where t.X > 5
                where t.X < 3
                select t;
    var solRelax = relax.Solve();
    Console.WriteLine($"-> {(solRelax == null ? "UNSAT (null)" : $"X = {solRelax.X}")}");
    bool bascule = solDur == null && solRelax != null && solRelax.X > 5;
    Console.WriteLine("Mesure : le verdict a bascule UNSAT -> SAT = " + bascule);
    if (!bascule)
        throw new InvalidOperationException(
            "Rewriter global : le verdict devait basculer UNSAT -> SAT (X > 5) -- verdict inchange.");
}


SANS rewriter : « X>5, X<3 » -> UNSAT (null)


AVEC rewriter global sur les memes contraintes ecrites :


  [rewriter global] 2 contrainte(s) recue(s)


  [rewriter global] 1 supprimee(s), 1 conservee(s)


-> X = 6


Mesure : le verdict a bascule UNSAT -> SAT = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture du résultat

Le rewriter global est un **transformateur de spécification** : il s'exécute une fois par
résolution, sur l'ensemble des contraintes, et ce qu'il rend *est* le problème posé au solveur.
Les usages canoniques : injecter des contraintes de domaine implicites (toutes les quantités sont
positives), dédupliquer des contraintes générées, appliquer une politique de relaxation,
journaliser la spécification effective.

Le prix est symétrique et doit être dit : **ce que vous écrivez n'est plus nécessairement ce que
le solveur résout**. D'où la section suivante — un rewriter global doit être documenté là où il
s'emploie, et le diagnostic doit rester lisible.


## 6. La cohérence `Explain`, montrée et non affirmée

`Explain()` rend un `SolveStatus` et, sur `UNSAT`, le **cœur insatisfiable** : le sous-ensemble de
contraintes qui participent réellement au conflit, chacune référencée par son index et son texte.

Le rewriter suivant **ajoute** au système une contrainte de domaine que l'appelant n'a jamais
écrite. Si le chemin était partagé, `Explain` doit voir cette contrainte injectée dans le cœur ;
s'il ne l'était pas, le cœur ne pourrait nommer que le `t.X < 0` d'origine, et le diagnostic
serait faux. C'est exactement le test que nous exécutons.


In [9]:
// Rewriter global qui AJOUTE une contrainte de domaine : "les quantites sont positives".
public class InjecteDomainePositif : ITheoremGlobalRewriter
{
    public IEnumerable<LambdaExpression> Rewrite(IEnumerable<LambdaExpression> constraints)
    {
        var liste = constraints.ToList();
        Expression<Func<AvecDomaine, bool>> domaine = t => t.X >= 0;
        Console.WriteLine($"  [injection] {liste.Count} contrainte(s) ecrite(s) + 1 contrainte de domaine (X >= 0)");
        return liste.Concat(new LambdaExpression[] { domaine });
    }
}

[TheoremGlobalRewriter(typeof(InjecteDomainePositif))]
public class AvecDomaine
{
    public int X { get; set; }
}

using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<AvecDomaine>()
                  where t.X < 0
                  select t;

    var sol = theorem.Solve();
    Console.WriteLine($"Solve()   : « X<0 » avec domaine injecte -> {(sol == null ? "UNSAT (null)" : $"X = {sol.X}")}");

    var explication = theorem.Explain();
    Console.WriteLine($"Explain() : statut = {explication.Status}, coeur de {explication.UnsatCore.Count} contrainte(s)");
    foreach (var c in explication.UnsatCore)
    {
        Console.WriteLine($"  {c}");
    }
    bool domaineCite = explication.UnsatCore.Any(c => c.Expression.Contains("X >= 0"));
    Console.WriteLine("Mesure : le coeur cite la contrainte INJECTEE par le rewriter = " + domaineCite);
    if (!domaineCite)
        throw new InvalidOperationException(
            "Explain() : le coeur devait citer la contrainte injectee (X >= 0) -- absente du coeur.");
}


  [injection] 1 contrainte(s) ecrite(s) + 1 contrainte de domaine (X >= 0)


Solve()   : « X<0 » avec domaine injecte -> UNSAT (null)


  [injection] 1 contrainte(s) ecrite(s) + 1 contrainte de domaine (X >= 0)


Explain() : statut = Unsatisfiable, coeur de 2 contrainte(s)


  #0: (t.X < 0)


  #1: (t.X >= 0)


Mesure : le coeur cite la contrainte INJECTEE par le rewriter = True



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



### Lecture du résultat

Le cœur insatisfiable nomme **deux** contraintes alors que l'appelant n'en a écrit qu'une : la
sienne, et celle que le rewriter a ajoutée. C'est la preuve visible que `Solve` et `Explain`
partagent le même chemin de contraintes — et la raison pour laquelle un rewriter global engage
aussi le diagnostic.

La règle d'usage qui en découle : un rewriter global se documente à l'endroit où il s'applique, et
son effet doit rester inspectable (ici, il s'annonce lui-même sur la sortie). Une spécification qui
change en silence rendrait le cœur insatisfiable ininterprétable pour la personne qui lit le
rapport d'échec.


## 7. À l'échelle : le rewriter global sur un Sudoku réel

Les sections 5 et 6 ont mesuré le rewriter global sur des systèmes-jouets d'une à deux
contraintes. Question légitime : **est-ce que ça tient à l'échelle ?** Cette section applique le
même mécanisme à l'encodage Sudoku du carnet 02 — 81 contraintes de domaine, 27 contraintes de
structure, 30 indices, soit **138 contraintes** — sur la même grille que le carnet 19 (30 indices
sur 81 cases).

Un rewriter global est une classe qui implémente `ITheoremGlobalRewriter`. Le binding l'instancie,
lui passe **toutes** les contraintes du théorème, et fait asserter ce qu'elle renvoie. Deux
conséquences importantes, toutes deux mesurées dans cette section :

- le rewriter travaille sur l'**arbre d'expressions**, avant toute traduction vers Z3 : il
  construit lui-même la contrainte qu'il ajoute ;
- l'attribut est lu sur le **type d'environnement** du théorème (`typeof(T)`) : un type
  d'environnement par configuration.

La règle que nous allons écrire est simple et vérifiable : **interdire la valeur 5 dans la
case 0**. La case 0 de la grille vaut exactement 5 (c'est un indice) : la règle contredit donc un
indice, et le théorème doit devenir insoluble.


In [10]:
// --- L'instance : la meme grille que le carnet 19 (30 indices sur 81 cases) ---
// Les lignes d'environnement alimentent les sorties commitees : un resultat sans
// environnement decrit n'est pas une preuve, c'est une anecdote.
const string Puzzle = "530070000600195000098000060800060003400803001700020006060000280000419005000080079";
Console.WriteLine($"Framework      : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
Console.WriteLine($"Architecture   : {System.Runtime.InteropServices.RuntimeInformation.ProcessArchitecture}");
Console.WriteLine($"Assembly Z3    : {typeof(Context).Assembly.GetName().Version}");
Console.WriteLine($"Indices de la grille : {Puzzle.Count(c => c != '0')} / 81 cases");

// --- Les parametres de la regle ---
// Le binding instancie le rewriter SANS argument : une regle parametrique se
// configure donc par un etat statique, lu au moment de la reecriture.
public static class Regle
{
    public static int CaseInterdite = 0;
    public static int ValeurInterdite = 5;
}

// --- Le rewriter global ---
// Il recoit toutes les contraintes, en ajoute une, et renvoie la nouvelle liste.
// La contrainte ajoutee est construite sur l'arbre d'expressions : on lit la
// propriete `Cells` du type d'environnement et on l'indexe.
public class ForbidValueRewriter : ITheoremGlobalRewriter
{
    public IEnumerable<LambdaExpression> Rewrite(IEnumerable<LambdaExpression> constraints)
    {
        var liste = constraints.ToList();
        var modele = liste[0];
        var t = modele.Parameters[0];
        var cases = Expression.Property(t, "Cells");
        var getItem = typeof(List<int>).GetMethod("get_Item")!;
        var caseInterdite = Expression.Call(cases, getItem, Expression.Constant(Regle.CaseInterdite));
        var corps = Expression.NotEqual(caseInterdite, Expression.Constant(Regle.ValeurInterdite));
        var ajoutee = Expression.Lambda(modele.Type, corps, t);
        Console.WriteLine($"  [ForbidValueRewriter] {liste.Count} contraintes recues ; ajout : {ajoutee.Body}");
        liste.Add(ajoutee);
        return liste;
    }
}

Console.WriteLine("Definitions pretes : Puzzle (30 indices), Regle (case 0, valeur 5 interdite), ForbidValueRewriter");


Framework      : .NET 10.0.14


Architecture   : X64


Assembly Z3    : 4.12.2.0


Indices de la grille : 30 / 81 cases


Definitions pretes : Puzzle (30 indices), Regle (case 0, valeur 5 interdite), ForbidValueRewriter



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



In [11]:
// --- Les types d'environnement ---
// Le binding lit l'attribut de reecriture sur le TYPE d'environnement : il faut
// donc un type par configuration. La propriete `Cells` doit etre declaree sur
// le type lui-meme (constate par une sonde hors carnet : heritee d'une classe
// de base, le binding echoue sur « Unknown parameter encountered: Cells. »).
public class SudokuArray
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

[TheoremGlobalRewriter(typeof(ForbidValueRewriter))]
public class SudokuArrayAvecRegle
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

// Un type d'environnement dont l'attribut pointe sur un type qui n'est PAS un
// rewriter global : la garde de formation de la section 8, preparee ici.
public class NotAGlobalRewriter { }

[TheoremGlobalRewriter(typeof(NotAGlobalRewriter))]
public class SudokuArrayGarde
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

Console.WriteLine("Types prets : SudokuArray (nu), SudokuArrayAvecRegle (avec regle), SudokuArrayGarde (fautif, section 8)");


Types prets : SudokuArray (nu), SudokuArrayAvecRegle (avec regle), SudokuArrayGarde (fautif, section 8)


### Les types d'environnement, et pourquoi il y en a plusieurs

Le binding lit l'attribut de réécriture sur le **type** de l'environnement : `typeof(T)` du
théorème. Un type par configuration, donc : `SudokuArray` pour l'énoncé nu, `SudokuArrayAvecRegle`
pour l'énoncé avec règle, `SudokuArrayGarde` pour le cas fautif de la section 8.

Chaque type déclare **sa propre** propriété `Cells`. Une propriété héritée d'une classe de base
échoue sur « Unknown parameter encountered: Cells. » (message constaté par une sonde hors carnet) :
la contrainte de conception du binding se paie des deux côtés — le type porte l'attribut, et il
porte sa propriété.


In [12]:
// --- L'encodage du carnet 02, repris tel quel ---
// Deux methodes presque identiques : le binding resout les membres de
// l'environnement sur le type CONCRET de l'arbre d'expressions. Un encodeur
// generique ne fonctionne pas (constate par une sonde hors carnet : « Unknown parameter encountered: Cells. »)
// : il faut donc un encodeur par type d'environnement.
public static class Sudoku09
{
    // 81 contraintes de domaine, 9 lignes, 9 colonnes, 9 boites, puis les indices.
    public static Theorem<SudokuArray> Charger(Z3Context ctx, string puzzle)
    {
        var th = ctx.NewTheorem<SudokuArray>();
        var indices = Enumerable.Range(0, 9).ToArray();

        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
            {
                var i1 = i; var j1 = j;
                th = th.Where(s => s.Cells[i1 * 9 + j1] > 0 && s.Cells[i1 * 9 + j1] < 10);
            }

        for (int r = 0; r < 9; r++)
        {
            var r1 = r;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(j => t.Cells[r1 * 9 + j]).ToArray()));
        }
        for (int c = 0; c < 9; c++)
        {
            var c1 = c;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(i => t.Cells[i * 9 + c1]).ToArray()));
        }
        for (int b = 0; b < 9; b++)
        {
            var b1 = b;
            var idx = (b1 / 3) * 3 * 9 + (b1 % 3) * 3;
            th = th.Where(t => Z3Methods.Distinct(new int[] {
                t.Cells[idx],      t.Cells[idx + 1],  t.Cells[idx + 2],
                t.Cells[idx + 9],  t.Cells[idx + 10], t.Cells[idx + 11],
                t.Cells[idx + 18], t.Cells[idx + 19], t.Cells[idx + 20] }));
        }

        for (int k = 0; k < 81; k++)
        {
            var k1 = k;
            var v = int.Parse(puzzle.Substring(k1, 1));
            if (v != 0) th = th.Where(t => t.Cells[k1] == v);
        }
        return th;
    }

    // Le meme encodage, sur le type d'environnement porte par l'attribut.
    public static Theorem<SudokuArrayAvecRegle> ChargerAvecRegle(Z3Context ctx, string puzzle)
    {
        var th = ctx.NewTheorem<SudokuArrayAvecRegle>();
        var indices = Enumerable.Range(0, 9).ToArray();

        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
            {
                var i1 = i; var j1 = j;
                th = th.Where(s => s.Cells[i1 * 9 + j1] > 0 && s.Cells[i1 * 9 + j1] < 10);
            }

        for (int r = 0; r < 9; r++)
        {
            var r1 = r;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(j => t.Cells[r1 * 9 + j]).ToArray()));
        }
        for (int c = 0; c < 9; c++)
        {
            var c1 = c;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(i => t.Cells[i * 9 + c1]).ToArray()));
        }
        for (int b = 0; b < 9; b++)
        {
            var b1 = b;
            var idx = (b1 / 3) * 3 * 9 + (b1 % 3) * 3;
            th = th.Where(t => Z3Methods.Distinct(new int[] {
                t.Cells[idx],      t.Cells[idx + 1],  t.Cells[idx + 2],
                t.Cells[idx + 9],  t.Cells[idx + 10], t.Cells[idx + 11],
                t.Cells[idx + 18], t.Cells[idx + 19], t.Cells[idx + 20] }));
        }

        for (int k = 0; k < 81; k++)
        {
            var k1 = k;
            var v = int.Parse(puzzle.Substring(k1, 1));
            if (v != 0) th = th.Where(t => t.Cells[k1] == v);
        }
        return th;
    }

    public static string Grille(List<int> cells) =>
        string.Join(" | ", Enumerable.Range(0, 9).Select(r =>
            string.Join("", Enumerable.Range(0, 9).Select(c => cells[r * 9 + c].ToString()))));
}

Console.WriteLine("Encodeur Sudoku09 pret : Charger (enonce nu) et ChargerAvecRegle (avec regle)");


Encodeur Sudoku09 pret : Charger (enonce nu) et ChargerAvecRegle (avec regle)



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### L'encodage, et pourquoi il est écrit deux fois

`Sudoku09.Charger` et `Sudoku09.ChargerAvecRegle` font presque la même chose : la même grille,
les mêmes contraintes de domaine, les mêmes contraintes de structure, les mêmes indices. La seule
différence est le type d'environnement retourné, qui porte — ou non — l'attribut de réécriture.

La duplication n'est pas une maladresse : le binding résout les membres de l'environnement sur le
**type concret** de l'arbre d'expressions. Un encodeur générique échoue sur
« Unknown parameter encountered: Cells. » (constaté par une sonde hors carnet) — la même contrainte
que pour la propriété héritée. Un point d'extension se paie donc aussi en contraintes de
conception.


In [13]:
Console.WriteLine("=== Reference : le meme enonce, sans reecrivain global");
using (var ctx = new Z3Context())
{
    var solution = Sudoku09.Charger(ctx, Puzzle).Solve();
    Console.WriteLine($"[DSL ] {Sudoku09.Grille(solution.Cells)}");
}

Console.WriteLine();
Console.WriteLine("=== Le meme enonce, avec le reecrivain global (interdit 5 en case 0)");
using (var ctx = new Z3Context())
{
    var solution = Sudoku09.ChargerAvecRegle(ctx, Puzzle).Solve();
    Console.WriteLine($"[DSL ] Solve -> {(solution == null ? "null (aucune solution)" : Sudoku09.Grille(solution.Cells))}");
}

Console.WriteLine();
Console.WriteLine("=== Le coeur UNSAT : quelles contraintes sont en conflit ?");
using (var ctx = new Z3Context())
{
    var explication = Sudoku09.ChargerAvecRegle(ctx, Puzzle).Explain();
    Console.WriteLine($"Statut : {explication.Status}");
    foreach (var c in explication.UnsatCore) Console.WriteLine($"  core : {c}");
}

Console.WriteLine();
Console.WriteLine($"Cases 0 et 71 : valeurs dans la grille = {Puzzle.Substring(0, 1)} et {Puzzle.Substring(71, 1)}");


=== Reference : le meme enonce, sans reecrivain global


[DSL ] 534678912 | 672195348 | 198342567 | 859761423 | 426853791 | 713924856 | 961537284 | 287419635 | 345286179


=== Le meme enonce, avec le reecrivain global (interdit 5 en case 0)


  [ForbidValueRewriter] 138 contraintes recues ; ajout : (s.Cells.get_Item(0) != 5)


[DSL ] Solve -> null (aucune solution)


...

  [ForbidValueRewriter] 138 contraintes recues ; ajout : (s.Cells.get_Item(0) != 5)


Statut : Unsatisfiable


  core : #108: (t.Cells.get_Item(value(Submission#13+Sudoku09+<>c__DisplayClass1_5).k1) == value(Submission#13+Sudoku09+<>c__DisplayClass1_5).v)


  core : #138: (s.Cells.get_Item(0) != 5)


Cases 0 et 71 : valeurs dans la grille = 5 et 5


### Lecture du résultat

Le même énoncé, résolu deux fois, donne **deux résultats différents** : une grille complète sans
le rewriter, `null` avec. La règle a changé le verdict, et elle l'a changé par un chemin
**visible** : le rewriter affiche la contrainte qu'il a construite, et le cœur UNSAT la nomme.

Le cœur UNSAT est la preuve la plus forte, parce qu'elle est **falsifiable** et **numérable** :

- le rewriter a reçu **138 contraintes** : 81 contraintes de domaine, 27 contraintes de structure
  (9 par ligne, 9 par colonne, 9 par boîte), et les 30 indices de la grille ;
- la contrainte qu'il a ajoutée est la **#138** : elle porte l'index 138 parce qu'elle vient après
  les 138 précédentes ;
- le cœur nomme la **#108** et la **#138** : la #108 est le premier indice de la grille
  (108 = 81 + 27), qui fixe la case 0 à 5 ; la #138 est la règle qui interdit 5 dans cette case.
  La ligne de sortie « Cases 0 et 71 » rappelle que la case 0 vaut bien 5 ; la case 71 est un
  **autre** indice 5, dont l'indice n'est **pas** dans le cœur : **#108 et #138 suffisent**
  à rendre le problème UNSAT — c'est là qu'on voit que le cœur est **minimal**.

Ce que ce carnet ne fait pas, c'est **inventer** une contradiction : l'index 0 de la grille vaut 5,
la règle interdit 5, le solveur ne peut pas satisfaire les deux. Un rewriter global n'est donc pas
une surcouche décorative : c'est une **contrainte réelle**, qui compte dans le cœur UNSAT comme les
contraintes écrites à la main.

La contrepartie est mesurée dans les cellules précédentes : l'encodage est **dupliqué** dans
`Sudoku09.Charger` et `Sudoku09.ChargerAvecRegle`. Ce n'est pas une maladresse du carnet, c'est une
propriété du binding : il résout les membres de l'environnement sur le **type concret** de l'arbre
d'expressions, si bien qu'un encodeur générique — ou une propriété héritée d'une classe de base —
échoue sur « Unknown parameter encountered: Cells. » (message constaté par une sonde hors carnet,
pas lu dans une sortie de ce carnet). Un point d'extension ne se paie pas seulement en lignes de
code : il se paie en **contraintes de conception**.


## 8. La troisième garde : l'attribut doit désigner une interface implémentée

La section 4 a mesuré deux refus du rewriter de prédicat (identité, cible non traduisible). Il en
manque une troisième, commune aux deux points d'extension : un attribut dont l'argument
`typeof(...)` pointe sur un type qui **n'implémente pas l'interface attendue** est refusé — pour
le rewriter de prédicat comme pour le rewriter global. Le binding vérifie le type **avant** de
l'instancier : il refuse donc un rewriter mal formé sans jamais lancer de code de votre part.

Ces messages ne sont pas des accidents : ce sont les **conditions du contrat** des deux interfaces,
écrites par le binding, et rapportées ici verbatim.


In [14]:
// --- Les declarations fautives : l'attribut designe un type qui n'implemente PAS l'interface ---
public class NotARewriter { }

public static class PredicatsFautifs
{
    [TheoremPredicateRewriter(typeof(NotARewriter))]
    public static bool CasInterface(int a, int b) => a != b;
}

Console.WriteLine("=== Garde : un type de rewriter de PREDICAT qui n'implemente pas l'interface");
try
{
    using var ctx = new Z3Context();
    Sudoku09.Charger(ctx, Puzzle)
        .Where(t => PredicatsFautifs.CasInterface(t.Cells[0], t.Cells[1]))
        .Solve();
    Console.WriteLine("[DSL ] aucune exception (inattendu)");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"[DSL ] InvalidOperationException : {ex.Message}");
}

Console.WriteLine();
Console.WriteLine("=== Garde : un type de rewriter GLOBAL qui n'implemente pas l'interface");
try
{
    using var ctx = new Z3Context();
    ctx.NewTheorem<SudokuArrayGarde>().Where(t => t.Cells[0] == 5).Solve();
    Console.WriteLine("[DSL ] aucune exception (inattendu)");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"[DSL ] InvalidOperationException : {ex.Message}");
}


=== Garde : un type de rewriter de PREDICAT qui n'implemente pas l'interface


[DSL ] InvalidOperationException : Invalid predicate rewriter type definition. Did you implement ITheoremPredicateRewriter?


=== Garde : un type de rewriter GLOBAL qui n'implemente pas l'interface


[DSL ] InvalidOperationException : Invalid global rewriter type definition. Did you implement ITheoremGlobalRewriter?



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



### Lecture du résultat

Les deux messages disent la même chose des deux points d'extension : l'attribut doit désigner un
type qui implémente l'interface attendue — `ITheoremPredicateRewriter` pour un attribut posé sur
une méthode, `ITheoremGlobalRewriter` pour un attribut posé sur un type d'environnement. Chacun
nomme l'interface manquante, et la vérification a lieu **avant** toute instanciation : le rewriter
mal formé ne s'exécute jamais.

Avec les deux témoins de la section 4, le tableau des gardes est complet : forme (interface
implémentée), effet (la réécriture doit avancer — garde anti-boucle), et grammaire (la cible doit
être traduisible). C'est **l'argument** qui rend le point d'extension utilisable en production :
le pouvoir d'écrire ses propres règles est un pouvoir **encadré**.


## 9. Les deux points d'extension, côte à côte

| | **Rewriter de prédicat** | **Rewriter global** |
|---|---|---|
| Attribut | `[TheoremPredicateRewriter]` sur une **méthode** | `[TheoremGlobalRewriter]` sur le **type d'environnement** |
| Déclencheur | chaque **appel** de cette méthode, pendant la traduction | une fois par **résolution** (et par `Explain`) |
| Entrée | le `MethodCallExpression` de l'appel | `IEnumerable<LambdaExpression>` — toutes les contraintes |
| Sortie | un appel à traduire **à la place** (doit différer de l'entrée) | les contraintes **à asserter** |
| Exigence | constructeur public sans paramètre | constructeur public sans paramètre |
| Garde mesurée | identité rejetée (anti-boucle), cible non traduisible rejetée, type non-interface rejeté (section 8) | type non-interface rejeté (section 8) ; pas de garde anti-boucle : la liberté inclut de vider le système |
| Contrainte de conception | aucune : la méthode annotée est une méthode ordinaire | l'attribut se lit sur `typeof(T)` : un type d'environnement par configuration, propriété déclarée sur le type lui-même (section 7) |
| Effet observé ici | un prédicat refusé devient résoluble | un `UNSAT` devient `SAT` ; un `SAT` devient `UNSAT` (sections 5-6) ; un Sudoku de 138 contraintes rendu insoluble par une règle (section 7) |
| Cas d'usage | vocabulaire métier : `TousDifferents`, `UneSeuleVraie`, `EstPair`… | domaines implicites, déduplication, relaxation, instrumentation, règle métier par état statique |
| Risque | cible hors grammaire du traducteur | l'énoncé résolu n'est plus celui écrit — documenter |

Les deux mécanismes sont **additifs** : aucune modification du fork n'est nécessaire, et le carnet
reste sur la build `.deploy/` partagée par toute la série.


## 10. Exercices

Stubs conformes C.1 : le carnet s'exécute de bout en bout même non complétés.


In [15]:
// EXERCICE 1 — Un troisieme predicat metier.
// Ecrivez PredicatsMetier.AuPlusUneVraie(bool a, bool b, bool c), annotez-la avec DeplieurMetier
// (ou un rewriter dedie), et faites-la correspondre a Z3Methods.AtMostOne(a, b, c).
// Resolvez ensuite : AuPlusUneVraie(P,Q,R) avec P et Q vrais -> attendu R = False.
// Indice : deux moities obligatoires. (1) La cible attend un params bool[] ;
// Expression.NewArrayInit construit ce tableau. (2) Si vous reutilisez DeplieurMetier,
// ajoutez AUSSI le case AuPlusUneVraie dans le switch de DeplieurMetier.Rewrite : son
// default leve "Predicat metier inconnu" pour tout appel non liste -- l'attribut seul
// ne suffit pas (cf. temoin negatif en debut de carnet). Un rewriter dedie evite cette
// deuxieme moitie.
// TODO etudiant
Console.WriteLine("Exercice 1 a completer : AuPlusUneVraie -> AtMostOne (stub C.1, aucune erreur volontaire)");


Exercice 1 a completer : AuPlusUneVraie -> AtMostOne (stub C.1, aucune erreur volontaire)


In [16]:
// EXERCICE 2 — Rewriter global de deduplication.
// Construisez un environnement dont les contraintes ecrites contiennent DEUX fois "t.X > 5",
// plus "t.X < 8". Ecrivez un rewriter global qui deduplique par c.Body.ToString(), affichez
// combien il en retire, et verifiez que la solution obtenue est inchangee.
// Question : pourquoi dedupliquer par ToString() est-il fragile des que le rewriter de predicat
// entre en jeu ?
// TODO etudiant
Console.WriteLine("Exercice 2 a completer : deduplication globale (stub C.1)");


Exercice 2 a completer : deduplication globale (stub C.1)


In [17]:
// EXERCICE 3 — Un rewriter global qui rend le systeme plus CONTRAIGNANT.
// Sur un environnement ou l'on ecrit seulement "t.X > 0", injectez "X != 7" et montrez que la
// solution evite 7. Puis rendez un systeme contradictoire par injection ("X < 0" plus injection
// "X >= 0") et verifiez que le coeur rendu par Explain() cite la contrainte injectee.
// Redigez en une phrase la regle d'usage : pourquoi un rewriter global doit-il etre documente
// dans le carnet qui l'emploie ?
// TODO etudiant
Console.WriteLine("Exercice 3 a completer : injection contraignante + lecture du coeur Explain (stub C.1)");


Exercice 3 a completer : injection contraignante + lecture du coeur Explain (stub C.1)


In [18]:
// EXERCICE 4 — Faire varier la regle Sudoku et lire le verdict (section 7).
// La regle de la section 7 vise la case 0 et interdit la valeur 5. Faites-la
// varier : c'est le rapport entre la valeur interdite et la solution qui decide
// si le theoreme reste soluble. Deux cas a distinguer :
//  (a) une regle qui interdit, sur la case visee, LA VALEUR QUE LA SOLUTION LUI
//      DONNE contredit cette solution -> le solveur ne rend plus de solution ;
//  (b) une regle qui interdit UNE AUTRE VALEUR ne contredit rien -> le solveur
//      rend toujours la meme solution.
// Attention : le statut de la case visee (indice ou case libre) n'est PAS le bon
// critere — le verdict ne depend que du rapport entre la valeur interdite et la
// valeur que la solution donne a la case.
// Indice : la regle se regle par les deux champs de Regle ; l'encodeur
// Sudoku09.ChargerAvecRegle fait le reste. La valeur que la solution donne a
// chaque case se lit sur la grille de reference de la section 7 ; les valeurs
// imposees se reperent aussi dans la chaine Puzzle (les chiffres non nuls).
// TODO etudiant
static string VerdictAvecRegle()
{
    // TODO etudiant : regler Regle.CaseInterdite / Regle.ValeurInterdite, resoudre
    // avec Sudoku09.ChargerAvecRegle, et renvoyer le verdict ("null" ou la grille)
    Console.WriteLine("Exercice a completer : regler la regle et renvoyer le verdict");
    return "a completer";
}

// La regle de la section 7 : case 0, valeur 5 interdite.
Regle.CaseInterdite = 0;
Regle.ValeurInterdite = 5;
Console.WriteLine($"Exercice 4 : verdict = {VerdictAvecRegle()}");


Exercice a completer : regler la regle et renvoyer le verdict


Exercice 4 : verdict = a completer


In [19]:
// EXERCICE 5 — Declencher une garde et rapporter son message verbatim.
// Les gardes du carnet sont reproductibles a la demande : identite (section 4),
// cible non traduisible (section 4), type non-interface pour le rewriter de
// predicat et pour le rewriter global (section 8). Reproduisez-en une, attrapez
// l'exception et renvoyez son message exact, sans que l'exception se propage.
// Indice : la garde "predicat non-interface" se declenche avec
// PredicatsFautifs.CasInterface ; la garde "global non-interface" avec le type
// d'environnement SudokuArrayGarde.
// TODO etudiant
static string MessageDeGarde(int numeroDeGarde)
{
    // TODO etudiant : appeler le theoreme qui declenche la garde demandee,
    // attraper l'InvalidOperationException, et renvoyer son message
    Console.WriteLine("Exercice a completer : renvoyer le message de la garde");
    return "a completer";
}

Console.WriteLine($"Exercice 5 : message = « {MessageDeGarde(1)} »");


Exercice a completer : renvoyer le message de la garde


Exercice 5 : message = « a completer »


## 11. Conclusion

Le DSL du fork n'est pas une boîte fermée : deux points d'extension permettent de l'adapter au
vocabulaire d'un domaine **sans toucher à la bibliothèque**.

- Le **rewriter de prédicat** apprend au traducteur un mot nouveau. Sa cible n'est pas du C#
  arbitraire : c'est une forme que la grammaire du traducteur couvre déjà — la démonstration
  négative de la section 4 le mesure, et c'est la contrainte à connaître avant d'en écrire un.
- Le **rewriter global** transforme la spécification entière avant assertion. Il change
  réellement le verdict (mesuré : `UNSAT` → `SAT` par suppression, `SAT` → `UNSAT` par injection),
  et `Explain` rapporte les contraintes **réécrites**, cœur insatisfiable à l'appui. À l'échelle
  (section 7) : un Sudoku de **138 contraintes** rendu insoluble par une seule règle ajoutée, le
  cœur UNSAT nommant la règle (#138) en face de l'indice contredit (#108).
- **Les deux mécanismes sont encadrés.** Type qui n'implémente pas l'interface (sections 4 et 8),
  rewriter qui ne réécrit rien : le binding refuse et le dit, avec un message qui nomme le
  coupable.
- **Un point d'extension a un coût de conception.** Le binding résout les membres de
  l'environnement sur le **type concret** de l'arbre d'expressions : l'encodage Sudoku a été
  dupliqué (`Charger` / `ChargerAvecRegle`) pour cette raison, et la propriété de l'environnement
  doit être déclarée sur le type lui-même — sinon « Unknown parameter encountered: Cells. ».

Le critère de bon usage est le même que pour toute extension : la réécriture doit préserver
l'intention de l'énoncé, et rester **visible** pour qui relit le carnet.
